In [2]:
pip install pymupdf


  Using cached pymupdf-1.25.5-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 209.5 kB/s eta 0:00:0000:0100:03
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Required libraries
import os
from pathlib import Path
import pandas as pd
import numpy as np
import fitz  # PyMuPDF
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from ipywidgets import FileUpload, VBox, Text, Output, Button
from IPython.display import display

# Setup folders
upload_folder = Path("./uploaded_papers")
upload_folder.mkdir(exist_ok=True, parents=True)

# UI Widgets
upload_widget = FileUpload(accept='.pdf', multiple=True)
search_box = Text(description='Search:')
search_button = Button(description='Search')
output = Output()

# Placeholder for paper metadata
all_papers = []

def extract_metadata_from_pdf(filepath):
    """
    Extracts title, authors, and domain from a PDF using metadata and text heuristics.
    """
    doc = fitz.open(filepath)
    meta = doc.metadata

    title = meta.get("title", "").strip()
    authors = meta.get("author", "").strip()

    # If no title/author in metadata, fallback to first page text
    if not title or not authors:
        text = doc[0].get_text()
        lines = text.split("\n")
        lines = [line.strip() for line in lines if line.strip()]

        if not title and lines:
            title = lines[0]  # assume first non-empty line is title

        if not authors and len(lines) > 1:
            # assume second line might be author line if not too long
            if len(lines[1].split()) <= 10:
                authors = lines[1]

    # Naive domain guess (optional: use LLM or classifier)
    domain = "Medical AI" if "health" in title.lower() or "medical" in title.lower() else "General AI"

    return title or "Untitled", authors or "Unknown Author", domain

# Upload handler
def on_upload_change(change):
    global all_papers
    all_papers = []

    for filename, fileinfo in upload_widget.value.items():
        filepath = upload_folder / filename
        with open(filepath, "wb") as f:
            f.write(fileinfo['content'])

        title, authors, domain = extract_metadata_from_pdf(filepath)

        all_papers.append({
            "title": title,
            "authors": authors,
            "domain": domain
        })

    # Load and chunk the PDFs
    loader = DirectoryLoader(upload_folder, show_progress=True)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=150,
        length_function=len,
        is_separator_regex=False,
    )
    pages = splitter.split_documents(documents)

    with output:
        output.clear_output()
        print(f"Uploaded {len(upload_widget.value)} file(s)")
        print(f"Total chunks: {len(pages)}")
        if pages:
            print("\n--- Sample Chunk ---\n")
            print(pages[0].page_content[:500], "...")

# Search handler
def search_papers(query):
    results = []
    for paper in all_papers:
        if query.lower() in paper['title'].lower() or \
           query.lower() in paper['authors'].lower() or \
           query.lower() in paper['domain'].lower():
            results.append(paper)
    return results

def on_search_clicked(b):
    q = search_box.value
    results = search_papers(q)
    with output:
        output.clear_output()
        if not results:
            print("No papers found.")
        else:
            print(f"Found {len(results)} paper(s):")
            for p in results:
                print(f"- {p['title']} by {p['authors']} [{p['domain']}]")

# Connect events
upload_widget.observe(on_upload_change, names='value')
search_button.on_click(on_search_clicked)

# Show UI
display(VBox([upload_widget, search_box, search_button, output]))


AttributeError: 'tuple' object has no attribute 'items'